# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/heyzara124-hub/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

Lane: **Refresh / Content Opportunity Scoring**. Mid-panel month used for all queries: `month=2026-03`.
The final month (June 2026 / `_sample` table) is never touched here — it stays a sealed test month.


## 0. Setup

Connect DuckDB to the Hugging Face warehouse release. Run this once at the top.

In [ ]:
%pip -q install duckdb huggingface_hub


In [ ]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Store it as a Colab Secret named HF_TOKEN (key icon on the left panel).
# Never paste the token directly into a cell -- this repo is public.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':   f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':   f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':    f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_march':    f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

# quick sanity check: this only reads Parquet metadata, so it is fast even though
# the full daily table has ~79M rows
for name in ['dim_clients', 'dim_content', 'fact_march']:
    n = con.sql(f"SELECT COUNT(*) FROM {TABLES[name]}").fetchone()[0]
    print(f'{name:12} {n:>12,} rows')


## 1. Unit of analysis + time window

**One row = one content item, for one client, on one report_date, inside March 2026.**

This is the daily fact table's own grain (`fact_content_daily_performance`), filtered down to one mid-panel month partition (`month=2026-03`). March 2026 is a full month sitting in the middle of the panel, not the sealed final month.

Time window: `report_date` between `2026-03-01` and `2026-03-31`, for every client and content item that has at least one row in that window.

In [ ]:
con.sql(f"SELECT * FROM {TABLES['fact_march']} LIMIT 5").df()


## 2. Fields: feature / label / context / excluded

**Feature** (safe to use, known before the decision point):
- `gsc_impressions`, `gsc_clicks` — daily search counts
- `gsc_avg_position` — daily search rank (0 means no data that day, not rank zero)
- `ga4_sessions` — daily engagement count, only kept where `ga4_data_available IS TRUE`

**Label / proxy** (the thing I predict, never a feature):
- `is_declining` — I build this myself: did impressions in the back half of March (Mar 16-31) drop by more than 20% versus the front half (Mar 1-15)

**Context** (for joining/grouping only, never a feature):
- `client_hash_id`, `content_hash_id`, `report_date`

**Excluded**, with why:
- Any row before a client's `ga4_data_start` — GA4 columns are zero-filled there, that is missing tracking, not zero engagement
- Product-decision fields like `health_score` or `priority_score` — not shipped in this table, and using them would just mean copying FlyRank's own answer instead of finding new signal

In [ ]:
# Supports the exclusion above: how many March rows actually have GA4 tracking on
con.sql(f"""
    SELECT ga4_data_available, COUNT(*) AS rows
    FROM {TABLES['fact_march']}
    GROUP BY 1
""").df()


## 3. Verify it with queries (grain, counts, missing values, windows)

Three required checks on the March partition, then the five-feature frame, then the leakage trap.

**Check 1 — grain**: one row really is one content item x client x day.

In [ ]:
# Empty result = the grain holds
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_march']}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()


**Check 2 — row count and date span** for this slice.

In [ ]:
con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT content_hash_id) AS content_items,
           COUNT(DISTINCT client_hash_id) AS clients,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {TABLES['fact_march']}
""").df()


**Check 3 — availability**, filtered with `IS TRUE`, showing how many rows survive.

In [ ]:
total = con.sql(f"SELECT COUNT(*) FROM {TABLES['fact_march']}").fetchone()[0]
available = con.sql(f"""
    SELECT COUNT(*) FROM {TABLES['fact_march']}
    WHERE ga4_data_available IS TRUE
""").fetchone()[0]
print(f'total rows:      {total:,}')
print(f'ga4 available:   {available:,}  ({available/total:.1%})')


### Five features (max)

Built from the **front half** of March only (Mar 1-15). The label comes from the **back half** (Mar 16-31), so nothing here can see its own answer.

In [ ]:
feature_frame = con.sql(f"""
    WITH per_item AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_fronthalf,
            SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END)      AS clk_fronthalf,
            AVG(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_avg_position > 0
                     THEN gsc_avg_position END)                                              AS avg_position_fronthalf,
            SUM(CASE WHEN report_date <= DATE '2026-03-15' AND ga4_data_available IS TRUE
                     THEN ga4_sessions ELSE 0 END)                                            AS sessions_fronthalf,
            SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_backhalf
        FROM {TABLES['fact_march']}
        GROUP BY 1, 2
        HAVING imp_fronthalf >= 50   -- minimum-volume filter, keeps out pure noise
    )
    SELECT *,
           CASE WHEN clk_fronthalf > 0 THEN clk_fronthalf::DOUBLE / imp_fronthalf ELSE 0 END AS ctr_fronthalf,
           CASE WHEN imp_backhalf < 0.8 * imp_fronthalf THEN 1 ELSE 0 END AS is_declining
    FROM per_item
""").df()

print(f'{len(feature_frame):,} content items with enough front-half volume')
feature_frame[['imp_fronthalf', 'clk_fronthalf', 'ctr_fronthalf',
               'avg_position_fronthalf', 'sessions_fronthalf', 'is_declining']].head()


The five features, each with a one-line "available when" answer:

1. **`imp_fronthalf`** — sum of impressions Mar 1-15 only. Available when: counted the moment Mar 15 ends.
2. **`clk_fronthalf`** — sum of clicks Mar 1-15 only. Available when: same cutoff, same reasoning.
3. **`ctr_fronthalf`** — clicks/impressions, both from the front half. Available when: computed only from numbers already known by Mar 15.
4. **`avg_position_fronthalf`** — average daily rank, front half, real ranks only (0 excluded). Available when: position is measured and logged same-day.
5. **`sessions_fronthalf`** — GA4 sessions, front half, only where GA4 was actually tracking. Available when: sessions are logged same-day, and the availability flag stops fake zeros from leaking in.

### The trap

Add one label-derived column on purpose (`imp_backhalf`, the exact number the label is computed from), train a quick model, watch the score jump toward perfect, then delete it.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

honest_cols = ['imp_fronthalf', 'clk_fronthalf', 'ctr_fronthalf',
               'avg_position_fronthalf', 'sessions_fronthalf']
model_data = feature_frame.dropna(subset=honest_cols + ['imp_backhalf'])
y = model_data['is_declining']

# --- WITH the leak: imp_backhalf sits right next to the label it defines ---
leaky_cols = honest_cols + ['imp_backhalf']
X_tr, X_te, y_tr, y_te = train_test_split(model_data[leaky_cols], y, test_size=0.25,
                                           random_state=42, stratify=y)
leaky_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
print(f'WITH leak (imp_backhalf included):  accuracy = {accuracy_score(y_te, leaky_model.predict(X_te)):.3f}')

# --- honest version: drop it ---
X_tr, X_te, y_tr, y_te = train_test_split(model_data[honest_cols], y, test_size=0.25,
                                           random_state=42, stratify=y)
honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
print(f'WITHOUT leak (honest features only): accuracy = {accuracy_score(y_te, honest_model.predict(X_te)):.3f}')
print(f'base rate (always predict majority):  {max(y_te.mean(), 1 - y_te.mean()):.3f}')


The leaky version scores near-perfect because `imp_backhalf` is literally the number `is_declining` is computed from -- the model is not finding a pattern, it is just reading the answer key. The honest number (features only, no back-half column) is the real number to report going forward.

## 4. Data limits

- **Unbalanced panel**: clients have different history lengths, and some have no GA4 tracking at all during March 2026 -- for those, engagement features are missing, not zero.
- **This is a proxy, not FlyRank's real decision**: `is_declining` only tests whether impressions dropped. It does not know why -- seasonality, an algorithm update, and a real content problem all look the same to this signal.
- **One month is short**: March 2026 alone will not catch slow or seasonal decline, only fast drops inside a 31-day window.

In [ ]:
# Supports the limitation above: how many clients have no GA4 tracking at all in March 2026
con.sql(f"""
    SELECT COUNT(DISTINCT client_hash_id) AS clients_with_ga4_in_march
    FROM {TABLES['fact_march']}
    WHERE ga4_data_available IS TRUE
""").df()


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.